In [0]:
#Load Data
from pyspark.sql.functions import *

errors = spark.table(
    "workspace.predictive_maintenance.silver_errors"
)

failures = spark.table(
    "workspace.predictive_maintenance.silver_failures"
)

In [0]:
#Check Data
print("Errors:", errors.count())
print("Failures:", failures.count())

display(errors.limit(5))
display(failures.limit(5))

Errors: 3919
Failures: 761


datetime,machineID,errorID
2015-01-27T04:00:00.000Z,1,error1
2015-08-30T01:00:00.000Z,1,error3
2015-05-30T20:00:00.000Z,2,error5
2015-08-18T22:00:00.000Z,2,error5
2015-12-28T06:00:00.000Z,2,error3


datetime,machineID,failure,failure_flag
2015-03-06T06:00:00.000Z,1,comp1,1
2015-01-19T06:00:00.000Z,10,comp2,1
2015-02-19T06:00:00.000Z,11,comp3,1
2015-07-10T06:00:00.000Z,13,comp2,1
2015-03-23T06:00:00.000Z,22,comp4,1


In [0]:
#Error Frequency
error_frequency = errors.groupBy(
    "errorID"
).count().orderBy(
    desc("count")
)

display(error_frequency)

errorID,count
error1,1010
error2,988
error3,838
error4,727
error5,356


Databricks visualization. Run in Databricks to view.

In [0]:
#Failures by Component
failure_frequency = failures.groupBy(
    "failure"
).count().orderBy(
    desc("count")
)

display(failure_frequency)

failure,count
comp2,259
comp1,192
comp4,179
comp3,131


Databricks visualization. Run in Databricks to view.

In [0]:
#Errors Before Failures
from pyspark.sql.functions import expr

error_failure_window = (
    errors.alias("e")
    .join(
        failures.alias("f"),
        (col("e.machineID") == col("f.machineID"))
        &
        (
            col("e.datetime")
            >=
            expr("f.datetime - INTERVAL 48 HOURS")
        )
        &
        (
            col("e.datetime")
            <=
            col("f.datetime")
        ),
        "inner"
    )
)

In [0]:
#Which Errors Occur Before Failures?
leading_errors = (
    error_failure_window
    .groupBy("errorID")
    .count()
    .orderBy(desc("count"))
)

display(leading_errors)

errorID,count
error3,318
error2,309
error1,237
error5,208
error4,169


Databricks visualization. Run in Databricks to view.

In [0]:
#Error vs Failed Component
error_component = (
    error_failure_window
    .groupBy(
        "errorID",
        "failure"
    )
    .count()
)

display(error_component)

errorID,failure,count
error5,comp4,178
error3,comp1,15
error2,comp1,11
error4,comp4,10
error2,comp2,262
error4,comp2,18
error4,comp3,130
error3,comp4,25
error5,comp3,9
error4,comp1,11


Databricks visualization. Run in Databricks to view.